# USD-M Futures Market Stream PoC — `fstream.binance.com`

USDT/USDC로 정산되는 무기한·만기 선물 시장이다. 세 가지 하위 카테고리가 있다.

```text
fstream.binance.com
 ├─ Crypto Perpetual   (암호화폐 무기한)   예: BTCUSDT, ETHUSDT
 ├─ TradFi Perpetual   (주식 무기한)       예: NVDAUSDT, MRVLUSDT
 └─ Delivery           (만기 선물)         Current Quarter / Next Quarter
```

공개 시장 데이터 수신에는 API 키가 필요 없다. Spot과 스트림 이름은 비슷하지만 페이로드 필드가 다르므로
같은 파서를 재사용하지 않는다.

- 관련 문서: [docs/binanceWebSocketREADME.md](../docs/binanceWebSocketREADME.md)


In [ ]:
%pip install -q "websockets>=17,<18"

In [ ]:
import asyncio
import json
import urllib.request

from websockets.asyncio.client import connect

WS_BASE = "wss://fstream.binance.com/ws"
REST_BASE = "https://fapi.binance.com"
SYMBOL = "btcusdt"

In [ ]:
async def receive(stream: str, message_count: int = 3, timeout: float = 15) -> list[dict]:
    url = f"{WS_BASE}/{stream}"
    messages = []

    async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        for _ in range(message_count):
            try:
                message = await asyncio.wait_for(ws.recv(), timeout=timeout)
            except TimeoutError:
                print(f"{timeout}초 동안 추가 메시지 없음: {len(messages)}건 수신")
                break
            messages.append(json.loads(message))

    return messages


def get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=10) as resp:
        return json.load(resp)

## 1. Crypto Perpetual — Trade Stream

Spot의 `trade`와 형태는 같지만 futures 전용 필드 `X`(주문 유형)가 추가된다.

- 스트림: `<symbol>@trade`
- 심볼: `BTCUSDT` (Crypto Perpetual)


In [ ]:
crypto_perp_trade_messages = await receive(f"{SYMBOL}@trade", message_count=3)
crypto_perp_trade_messages

## 2. Crypto Perpetual — Kline Stream

- 스트림: `<symbol>@kline_<interval>`


In [ ]:
KLINE_INTERVAL = "1m"

crypto_perp_kline_messages = await receive(
    f"{SYMBOL}@kline_{KLINE_INTERVAL}",
    message_count=3,
)
crypto_perp_kline_messages

## 3. Crypto Perpetual — Book Ticker Stream

Spot의 `bookTicker`와 달리 futures는 이벤트 타입(`e`)과 시간 필드(`E`, `T`)가 포함된다.

- 스트림: `<symbol>@bookTicker`


In [ ]:
crypto_perp_book_ticker_messages = await receive(f"{SYMBOL}@bookTicker", message_count=3)
crypto_perp_book_ticker_messages

## 4. Crypto Perpetual — Mark Price Stream (마크 가격 · 펀딩비)

무기한 선물에만 있는 스트림이다. 청산 기준가인 마크 가격과 현재 펀딩비율(`r`)을 전달한다.
펀딩비가 양수면 롱이 숏에게 지불한다 = 롱 포지션이 과열됐다는 신호다.

- 스트림: `<symbol>@markPrice` (1초 갱신은 `<symbol>@markPrice@1s`)
- 주요 필드: `p` markPrice, `i` indexPrice, `r` fundingRate, `T` nextFundingTime,
  `ap` markPrice의 이동평균(mark price moving average),
  `st` symbolType — CM 마이그레이션 이후 필드, `1`=UM(USDⓈ-M), `2`=CM(COIN-M)
- 네트워크/방화벽에 따라 이 스트림만 지연되거나 도착하지 않는 경우가 있었다. 응답이 비어 있으면
  방화벽에서 해당 프레임을 차단하고 있을 가능성이 있으니 다른 네트워크에서 재시도한다.


In [ ]:
mark_price_messages = await receive(f"{SYMBOL}@markPrice", message_count=1, timeout=20)
mark_price_messages

## 5. Crypto Perpetual — Force Order Stream (강제 청산)

대량 강제 청산 이벤트다. 개별 심볼 스트림은 이벤트 자체가 드물어 전체 심볼을 묶은 `!forceOrder@arr`로
확인한다. 청산이 활발하지 않은 시간대에는 타임아웃으로 빈 리스트가 반환될 수 있다(정상 동작이다).

- 스트림: `!forceOrder@arr` (전체 심볼) 또는 `<symbol>@forceOrder`
- 주요 필드: `o.s` 심볼, `o.S` 청산 방향, `o.q` 청산 수량, `o.ap` 평균 체결가


In [ ]:
force_order_messages = await receive("!forceOrder@arr", message_count=1, timeout=45)
force_order_messages

## 6. TradFi Perpetual — NVDAUSDT Trade Stream

NVIDIA 주가를 기초로 하는 Binance의 USDT 결제형 무기한 선물이다. 실제 Nasdaq 주식 현물이 아니라
`TRADFI_PERPETUAL` 파생상품이며, 스트림 구조는 Crypto Perpetual과 동일하다.

- 심볼: `NVDAUSDT`


In [ ]:
tradfi_perp_trade_messages = await receive("nvdausdt@trade", message_count=3)
tradfi_perp_trade_messages

## 7. TradFi Perpetual — MRVLUSDT Trade Stream

같은 카테고리의 다른 심볼로, 호출 방식은 동일하다.

- 심볼: `MRVLUSDT`


In [ ]:
tradfi_perp_trade_messages_2 = await receive("mrvlusdt@trade", message_count=3)
tradfi_perp_trade_messages_2

## 8. Delivery — Current/Next Quarter 심볼 조회

만기 선물의 심볼은 만기가 돌아올 때마다 바뀐다(예: `BTCUSDT_260925`). 하드코딩하지 않고
`exchangeInfo`에서 `contractType`으로 현재/다음 분기 심볼을 매번 조회한다.

- REST: `GET /fapi/v1/exchangeInfo`
- 필터: `pair == SYMBOL.upper()` and `contractType in (CURRENT_QUARTER, NEXT_QUARTER)`


In [ ]:
exchange_info = get_json(f"{REST_BASE}/fapi/v1/exchangeInfo")
quarterlies = [
    s for s in exchange_info["symbols"]
    if s.get("pair") == SYMBOL.upper() and s.get("contractType") in ("CURRENT_QUARTER", "NEXT_QUARTER")
]
[(s["symbol"], s["contractType"], s["status"]) for s in quarterlies]

In [ ]:
current_quarter = next(s["symbol"] for s in quarterlies if s["contractType"] == "CURRENT_QUARTER")
current_quarter

## 9. Delivery — Current Quarter Book Ticker

만기 선물은 거래량이 적어 `trade`/`kline`이 한동안 오지 않을 수 있다. 최우선 호가(`bookTicker`)는
체결 여부와 무관하게 갱신되므로 PoC로 더 안정적이다.


In [ ]:
delivery_book_ticker_messages = await receive(f"{current_quarter.lower()}@bookTicker", message_count=3)
delivery_book_ticker_messages

## 10. Combined Streams

Crypto Perpetual의 `trade` + `kline_1m` + `bookTicker`를 하나의 연결로 묶어서 수신한다.


In [ ]:
async def receive_combined(streams: list[str], message_count: int = 5) -> list[dict]:
    stream_names = "/".join(streams)
    url = f"wss://fstream.binance.com/stream?streams={stream_names}"
    messages = []

    async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        for _ in range(message_count):
            messages.append(json.loads(await ws.recv()))

    return messages


combined_messages = await receive_combined(
    [f"{SYMBOL}@trade", f"{SYMBOL}@kline_1m", f"{SYMBOL}@bookTicker"],
    message_count=5,
)
combined_messages